# TF-IDF + Logistic Regression baseline: the whole record

One notebook that reproduces the complete Week 1 baseline and exports it for a website:

1. Rebuild `master.csv` from the raw files and check it matches the one the team uses
2. Rebuild the 70/15/15 split and check it matches `split.csv`
3. Train TF-IDF + Logistic Regression (v0, raw text)
4. Evaluate: accuracy, precision, recall, F1 (Fake = positive class), confusion matrix, classification report
5. Predictions file for AI/ML-03, and a check against the official one
6. Top Fake / Real terms (the publisher-style shortcut), and a check against the official files
7. Confidence calibration, and the one-word `(Reuters)` rule for comparison
8. Optional: v1 on AI/ML-01's cleaned text
9. Save everything, including **one pickle** (`fake_news_baseline.pkl`) for a website

**Nothing official is overwritten.** Outputs go to `models/baseline/export/` (gitignored) unless you set `WRITE_TO_OFFICIAL = True` in the first cell.
Run locally from the `notebooks/` folder. On Kaggle only steps 3 to 9 run (the raw files are not there).

In [ ]:
import json, pickle
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
FAKE_LABEL, REAL_LABEL = 1, 0
WRITE_TO_OFFICIAL = False   # True = also write models/baseline/v0_raw and results/baseline (overwrites the official files)

ON_KAGGLE = Path("/kaggle/input").exists()
if ON_KAGGLE:
    DATA = Path("/kaggle/input/datasets/govindvijay001/fake-detection")
    ROOT, RAW = None, None
    MASTER, SPLIT = DATA / "master.csv", DATA / "split.csv"
    EXPORT = Path("/kaggle/working/baseline_export")
    OFFICIAL_RES = None
else:
    ROOT = Path("..").resolve() if Path("../data").exists() else Path(".").resolve()
    RAW = ROOT / "data" / "raw"
    MASTER, SPLIT = ROOT / "data" / "interim" / "master.csv", ROOT / "data" / "splits" / "split.csv"
    EXPORT = ROOT / "models" / "baseline" / "export"
    OFFICIAL_RES = ROOT / "results" / "baseline"

if WRITE_TO_OFFICIAL and ROOT is not None:
    MODEL_OUT, RES_OUT = ROOT / "models" / "baseline" / "v0_raw", ROOT / "results" / "baseline"
else:
    MODEL_OUT, RES_OUT = EXPORT, EXPORT / "results"
for d in (EXPORT, MODEL_OUT, RES_OUT):
    d.mkdir(parents=True, exist_ok=True)

checks = {}   # reproduction checks, summarised in the last cell
print("project root:", ROOT)
print("outputs go to:", MODEL_OUT, "and", RES_OUT)
print("scikit-learn", sklearn.__version__)

## 1. Rebuild `master.csv` from the raw files (in memory)

Same steps as `src/data_loading.py`: labels, blank-body removal, `raw_text = title + ". " + body`, drop articles labelled both ways, drop duplicates, add `row_id`.

In [ ]:
def build_master_from_raw(raw_dir):
    fake = pd.read_csv(raw_dir / "Fake.csv"); fake["label"] = FAKE_LABEL
    real = pd.read_csv(raw_dir / "True.csv"); real["label"] = REAL_LABEL
    df = pd.concat([fake, real], ignore_index=True)
    print("raw rows:", len(df), "| Fake", len(fake), "| Real", len(real))

    df["title"] = df["title"].fillna("")
    df["text"] = df["text"].fillna("")
    blank = df["text"].str.strip().str.len() == 0
    print("blank bodies:", int(blank.sum()), "(by label:", df[blank]["label"].value_counts().to_dict(), ")")
    df = df[~blank]
    print("after removing blank text:", len(df))

    df["raw_text"] = (df["title"] + ". " + df["text"]).str.strip()
    both = df.groupby("raw_text")["label"].nunique()
    both = both[both > 1].index
    print("texts labelled both Fake and Real:", len(both))
    df = df[~df["raw_text"].isin(both)]
    df = df.drop_duplicates(subset="raw_text").reset_index(drop=True)
    print("after removing duplicates:", len(df))
    df.insert(0, "row_id", range(len(df)))
    return df

existing = pd.read_csv(MASTER, usecols=["row_id", "label", "raw_text"])
if RAW is not None and (RAW / "Fake.csv").exists() and (RAW / "True.csv").exists():
    built = build_master_from_raw(RAW)
    checks["master.csv reproduced from raw files"] = bool(
        len(built) == len(existing)
        and (built["row_id"].to_numpy() == existing["row_id"].to_numpy()).all()
        and (built["label"].to_numpy() == existing["label"].to_numpy()).all()
        and (built["raw_text"].astype(str).to_numpy() == existing["raw_text"].astype(str).to_numpy()).all())
    print("matches the existing master.csv:", checks["master.csv reproduced from raw files"])
else:
    print("Raw files not found (e.g. on Kaggle): using the existing master.csv as is.")
print("master.csv:", len(existing), "articles | labels (0 = Real, 1 = Fake):", existing["label"].value_counts().to_dict())

## 2. Rebuild the split and check it matches `split.csv`

Stratified 70/15/15, seed 42, exactly as in `src/make_splits.py`. Training below uses the official `split.csv`.

In [ ]:
ids = existing[["row_id", "label"]]
train_val, test_ids = train_test_split(ids, test_size=0.15, stratify=ids["label"], random_state=RANDOM_STATE)
train_ids, val_ids = train_test_split(train_val, test_size=0.15 / 0.85,
                                      stratify=train_val["label"], random_state=RANDOM_STATE)
rebuilt = (pd.concat([train_ids.assign(split="train"), val_ids.assign(split="val"),
                      test_ids.assign(split="test")])[["row_id", "split"]]
           .sort_values("row_id").reset_index(drop=True))
official_split = pd.read_csv(SPLIT).sort_values("row_id").reset_index(drop=True)
checks["split.csv reproduced"] = bool(rebuilt.equals(official_split))
print("split sizes:", official_split["split"].value_counts().to_dict())
print("matches the official split.csv:", checks["split.csv reproduced"])

## 3. Load the data and train (v0, raw text)

In [ ]:
df = existing.merge(official_split, on="row_id", how="left")
assert df["split"].notna().all(), "some row_ids are missing from split.csv"
train, val, test = (df[df["split"] == s] for s in ("train", "val", "test"))
print("train", len(train), "| val", len(val), "| test", len(test))

def make_pipeline():
    return Pipeline([
        ("tfidf", TfidfVectorizer(max_features=50_000, ngram_range=(1, 2),
                                  min_df=5, max_df=0.9, sublinear_tf=True)),
        ("clf", LogisticRegression(max_iter=1000, C=1.0, random_state=RANDOM_STATE)),
    ])

pipe = make_pipeline()
pipe.fit(train["raw_text"], train["label"])        # vectorizer fitted on the training split only
vec, clf = pipe.named_steps["tfidf"], pipe.named_steps["clf"]
print("features:", len(vec.get_feature_names_out()), "| classes:", clf.classes_, "(1 = Fake)")

## 4. Evaluate (Fake = positive class)

In [ ]:
def four_metrics(y, pred):
    return {"accuracy": round(accuracy_score(y, pred), 4),
            "precision_fake": round(precision_score(y, pred), 4),
            "recall_fake": round(recall_score(y, pred), 4),
            "f1_fake": round(f1_score(y, pred), 4)}

metrics = {"val": four_metrics(val["label"], pipe.predict(val["raw_text"])),
           "test": four_metrics(test["label"], pipe.predict(test["raw_text"]))}
print(json.dumps(metrics, indent=2))

test_pred = pipe.predict(test["raw_text"])
print("\nTest classification report:")
print(classification_report(test["label"], test_pred, target_names=["Real", "Fake"], digits=4))
print("Test confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(test["label"], test_pred))

if OFFICIAL_RES is not None and (OFFICIAL_RES / "metrics_v0_raw.json").exists():
    official_metrics = json.loads((OFFICIAL_RES / "metrics_v0_raw.json").read_text())
    shared = [k for k in metrics["test"] if k in official_metrics["test"]]
    checks["metrics match metrics_v0_raw.json"] = all(
        abs(metrics[s][k] - official_metrics[s][k]) < 1e-4 for s in ("val", "test") for k in shared)
    print("\nmatches the official metrics_v0_raw.json:", checks["metrics match metrics_v0_raw.json"])

## 5. Predictions for AI/ML-03 (val + test) and a check against the official file

In [ ]:
parts = []
for part in (val, test):
    parts.append(pd.DataFrame({
        "row_id": part["row_id"].values, "split": part["split"].values,
        "y_true": part["label"].values, "y_pred": pipe.predict(part["raw_text"]),
        "prob_fake": pipe.predict_proba(part["raw_text"])[:, 1]}))
predictions = pd.concat(parts, ignore_index=True)
print(predictions.shape, predictions.columns.tolist(), "| row_id unique:", predictions["row_id"].is_unique)

if OFFICIAL_RES is not None and (OFFICIAL_RES / "predictions_v0_raw.csv").exists():
    off = pd.read_csv(OFFICIAL_RES / "predictions_v0_raw.csv")
    m = predictions.merge(off, on="row_id", suffixes=("", "_off"))
    checks["predictions match predictions_v0_raw.csv"] = bool(
        len(m) == len(off) == len(predictions)
        and (m["y_pred"] == m["y_pred_off"]).all()
        and np.abs(m["prob_fake"] - m["prob_fake_off"]).max() < 1e-6)
    print("same predictions as the official file:", checks["predictions match predictions_v0_raw.csv"])

## 6. Top terms: what the model relies on

If publisher-style words such as `reuters` or `said` lead the Real side, and `video`, `via`, `featured image` lead the Fake side,
the model is learning source style, not truthfulness.

In [ ]:
names = vec.get_feature_names_out()
top = pd.DataFrame({"term": names, "weight": clf.coef_[0]}).sort_values("weight")
top_real = top.head(25).assign(pushes_toward="REAL")
top_fake = top.tail(25).iloc[::-1].assign(pushes_toward="FAKE")
print("Strongest Real terms:\n", top_real.head(10).to_string(index=False))
print("\nStrongest Fake terms:\n", top_fake.head(10).to_string(index=False))

if OFFICIAL_RES is not None and (OFFICIAL_RES / "top_real_terms_v0_raw.csv").exists():
    off_r = pd.read_csv(OFFICIAL_RES / "top_real_terms_v0_raw.csv")["term"].tolist()
    off_f = pd.read_csv(OFFICIAL_RES / "top_fake_terms_v0_raw.csv")["term"].tolist()
    checks["top terms match the official files"] = bool(off_r == top_real["term"].tolist()
                                                        and off_f == top_fake["term"].tolist())
    print("\nsame top terms as the official files:", checks["top terms match the official files"])

## 7. Confidence calibration and the one-word rule

**Confidence** is the model's probability for its chosen label, `max(prob_fake, 1 - prob_fake)`. It is not accuracy.
The table shows how often the model is right inside each confidence range on the test split.
The **rule** predicts Real if the article contains `(Reuters)`, otherwise Fake. It needs no training.

In [ ]:
t = predictions[predictions["split"] == "test"].copy()
t["confidence"] = t["prob_fake"].apply(lambda p: max(p, 1 - p))
t["correct"] = t["y_true"] == t["y_pred"]
t["bucket"] = pd.cut(t["confidence"], [0.5, 0.7, 0.9, 0.99, 1.0])
print(t.groupby("bucket", observed=True)["correct"].agg(["mean", "count"]))

test_text = test.set_index("row_id").loc[t["row_id"], "raw_text"]
rule_pred = (~test_text.str.contains("(Reuters)", regex=False)).astype(int).to_numpy()
rows = []
for name, pred in [("Rule: '(Reuters)' = Real, else Fake", rule_pred),
                   ("TF-IDF + Logistic Regression", t["y_pred"].to_numpy())]:
    tn, fp, fn, tp = confusion_matrix(t["y_true"], pred, labels=[0, 1]).ravel()
    rows.append({"model": name, **four_metrics(t["y_true"], pred), "FP": fp, "FN": fn, "errors": fp + fn})
print()
print(pd.DataFrame(rows).to_string(index=False))

## 8. Optional: v1 on AI/ML-01's cleaned text

Runs only if `data/processed/cleaned_dataset.csv` exists. It covers the subset of articles in that file, so v0 is scored on the **same test articles** for a fair comparison.
That cleaned text appears to remove source cues, so this measures "AI/ML-01's cleaned text", not "the same model on cleaned text".

In [ ]:
CLEANED = (ROOT / "data" / "processed" / "cleaned_dataset.csv") if ROOT is not None else None
if CLEANED is not None and CLEANED.exists():
    c = pd.read_csv(CLEANED).merge(official_split, on="row_id", how="inner")
    c["cleaned_text"] = c["cleaned_text"].fillna("")
    c_train, c_test = c[c["split"] == "train"], c[c["split"] == "test"]
    pipe_v1 = make_pipeline().fit(c_train["cleaned_text"], c_train["label"])
    pred_v1 = pipe_v1.predict(c_test["cleaned_text"])

    same_rows = test[test["row_id"].isin(c_test["row_id"])].set_index("row_id").loc[c_test["row_id"]]
    pred_v0 = pipe.predict(same_rows["raw_text"])
    print("v1 uses", len(c), "of", len(df), "articles | common test articles:", len(c_test))
    for name, pred in [("v0 (raw text)", pred_v0), ("v1 (cleaned text)", pred_v1)]:
        print(name, four_metrics(c_test["label"], pred), "errors", int((pred != c_test["label"].to_numpy()).sum()))
else:
    print("cleaned_dataset.csv not found: skipping v1.")

## 9. Save everything

The pickle is **one dict** (pipeline, label map, threshold, input format, scikit-learn version, test metrics) for the website.
The vectorizer's `stop_words_` attribute is cleared first: it only lists pruned terms, is not needed for prediction, and can make the file very large.

In [ ]:
if getattr(vec, "stop_words_", None) is not None:
    vec.stop_words_ = None

joblib.dump(vec, MODEL_OUT / "tfidf_vectorizer.joblib")
joblib.dump(clf, MODEL_OUT / "logreg_model.joblib")
(RES_OUT / "metrics_v0_raw.json").write_text(json.dumps(metrics, indent=2))
predictions.to_csv(RES_OUT / "predictions_v0_raw.csv", index=False)
top_real.to_csv(RES_OUT / "top_real_terms_v0_raw.csv", index=False)
top_fake.to_csv(RES_OUT / "top_fake_terms_v0_raw.csv", index=False)

bundle = {
    "pipeline": pipe,
    "label_map": {0: "Real", 1: "Fake"},
    "threshold": 0.5,
    "input_format": "title + '. ' + body, one string (same as raw_text)",
    "trained_on": "master.csv train split (v0_raw baseline)",
    "sklearn_version": sklearn.__version__,
    "test_metrics": metrics["test"],
}
PKL = EXPORT / "fake_news_baseline.pkl"
with open(PKL, "wb") as f:
    pickle.dump(bundle, f)
print("models saved in:", MODEL_OUT)
print("results saved in:", RES_OUT)
print("pickle:", PKL, "| %.1f MB" % (PKL.stat().st_size / 1e6))

with open(PKL, "rb") as f:
    loaded = pickle.load(f)
sample = test["raw_text"].head(200)
checks["reloaded pickle gives identical probabilities"] = bool(np.allclose(
    pipe.predict_proba(sample)[:, 1], loaded["pipeline"].predict_proba(sample)[:, 1]))

## 10. How the website uses the pickle

Copy `predict_text` into the website's code. It needs the same scikit-learn version (stored in the pickle). Only unpickle files you created yourself: unpickling runs code.
The output has the same format as `src/predictor.py`.

In [ ]:
def predict_text(bundle, text):
    if not isinstance(text, str) or not text.strip():
        return {"label": "Unknown", "confidence": None, "prob_fake": None, "error": "Empty input"}
    prob_fake = float(bundle["pipeline"].predict_proba([text])[0, 1])
    out = {"label": bundle["label_map"][int(prob_fake >= bundle["threshold"])],
           "confidence": max(prob_fake, 1 - prob_fake), "prob_fake": prob_fake}
    if len(text.split()) < 30:
        out["warning"] = "Very short text. The model was trained on full news articles, so this result is unreliable."
    return out

for text in ["WASHINGTON (Reuters) - The U.S. Senate voted on Tuesday to approve a stopgap spending bill, "
             "averting a government shutdown, officials said. The measure passed and now goes to the House "
             "of Representatives for a vote later this week, according to congressional aides.",
             "Watch this video, featured image via Twitter. Read more!", ""]:
    print(predict_text(loaded, text))

## 11. Summary of the reproduction checks

In [ ]:
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", "|", name)
if not checks:
    print("No checks ran.")
print()
print("Test metrics:", metrics["test"])
print("Reference (Week 1): accuracy 0.9864, precision 0.9919, recall 0.9779, F1 0.9848")